In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import numpy as np

from torch.utils.data import DataLoader
from torch.utils.data import Dataset


In [6]:
class RefNRIMLP(nn.Module):
    """Two-layer fully-connected ELU net with optional batch norm."""

    def __init__(self, n_in, n_hid, n_out, dropout_prob=0., no_bn=False):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_in, n_hid),
            nn.ELU(inplace=True),
            nn.Dropout(dropout_prob),
            nn.Linear(n_hid, n_out),
            nn.ELU(inplace=True),
            nn.BatchNorm1d(n_out) if not no_bn else nn.Identity()
        )

        # Initialize weights
        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                m.bias.data.fill_(0.1)
            elif isinstance(m, nn.BatchNorm1d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

    def forward(self, x):
        # Reshape if needed for BatchNorm
        orig_shape = x.shape
        if len(orig_shape) > 2:
            x = x.view(-1, orig_shape[-1])

        x = self.net(x)

        # Restore original shape if needed
        if len(orig_shape) > 2:
            x = x.view(*orig_shape[:-1], -1)

        return x

In [7]:
class BaseEncoder(nn.Module):
    def __init__(self, num_vars):
        super(BaseEncoder, self).__init__()
        self.num_vars = num_vars
        # creating a matrix representing edges between variables
        edges = torch.ones(num_vars) - torch.eye(num_vars)
        # finds the indicies where edges exist
        self.send_edges, self.recv_edges = torch.where(edges)

        # one-hot representation for receiving edges
        one_hot_recv = torch.nn.functional.one_hot(
            self.recv_edges,
            num_classes=num_vars,
        )
        # tensor parameter for edge to node transformations
        self.edge2node_mat = nn.Parameter(
            one_hot_recv.float().T,
            requires_grad=False,
        )

    # gets sender and receiver embeddings
    def node2edge(self, node_embeddings):
        send_embed = node_embeddings[:, self.send_edges]
        recv_embed = node_embeddings[:, self.recv_edges]
        # concatenates sender and receiver embeddings
        return torch.cat([send_embed, recv_embed], dim=2)

    def edge2node(self, edge_embeddings):
        # multiplies edge embeddings with the edge to node matrix to get incoming messages for each node
        incoming = torch.matmul(self.edge2node_mat, edge_embeddings)
        # normalizes the incoming embedding
        return incoming / (self.num_vars - 1)
    
    def forward(self, inputs, state=None, return_state=False):
        raise NotImplementedError

In [8]:
class RefMLPEncoder(BaseEncoder):
    def __init__(self,
            num_vars=31,
            input_size=6,
            input_time_steps=50,
            encoder_mlp_hidden=256,
            encoder_hidden=256,
            num_edge_types=2,
            encoder_dropout=0.):
        super(RefMLPEncoder, self).__init__(num_vars)
        inp_size = input_size * input_time_steps
        hidden_size = encoder_hidden
        num_layers = 3
        self.input_time_steps = input_time_steps

        # Defines MLP layers. RefNRIMLP is a 2-layer fully connected ELU net with batch norm.
        self.mlp1 = RefNRIMLP(inp_size, hidden_size, hidden_size, encoder_dropout)
        self.mlp2 = RefNRIMLP(hidden_size*2, hidden_size,hidden_size, encoder_dropout)
        self.mlp3 = RefNRIMLP(hidden_size, hidden_size,hidden_size, encoder_dropout)
        mlp4_inp_size = hidden_size * 2
        self.mlp4 = RefNRIMLP(mlp4_inp_size, hidden_size,hidden_size, encoder_dropout)

        # Defines our fully connected layer
        layers = [nn.Linear(hidden_size, encoder_mlp_hidden), nn.ELU(inplace=True)]
        layers += [nn.Linear(encoder_mlp_hidden, encoder_mlp_hidden),
                  nn.ELU(inplace=True)] * (num_layers - 2)
        layers.append(nn.Linear(encoder_mlp_hidden, num_edge_types))
        self.fc_out = nn.Sequential(*layers)
        self.init_weights()

    def node2edge(self, node_embeddings):
        send_embed = node_embeddings[:, self.send_edges, :]
        recv_embed = node_embeddings[:, self.recv_edges, :]
        return torch.cat([send_embed, recv_embed], dim=2)

    def edge2node(self, edge_embeddings):
        incoming = torch.matmul(self.edge2node_mat, edge_embeddings)
        return incoming/(self.num_vars-1)

    def init_weights(self):
        for m in self.modules():
            # Only applies to linear layers
            if isinstance(m, nn.Linear):
                # initialize the weights using the Xavier initialization approach. 
                # This ensures that the gradients in the layers are all approximately of similar scale,
                # which reduces the risk of our loss rapidly diverging—known as blow-up
                nn.init.xavier_normal_(m.weight.data)
                # We also set the initial bias to 0.1, 
                # which further helps with the stability of training.
                m.bias.data.fill_(0.1)

    def merge_states(self, states):
        return torch.cat(states, dim=0)

    def forward(self, inputs, state=None, return_state=False):
        if inputs.size(1) > self.input_time_steps:
            inputs = inputs[:, -self.input_time_steps:]
        elif inputs.size(1) < self.input_time_steps:
            begin_inp = inputs[:, 0:1].expand(
            -1,
            self.input_time_steps-inputs.size(1),
            -1, -1
            )
            inputs = torch.cat([begin_inp, inputs], dim=1)

        # New shape: [num_sims, num_atoms, num_timesteps*num_dims]
        x = inputs.transpose(1, 2).contiguous()
        x = x.view(inputs.size(0), inputs.size(2), -1)

        # Passes through first MLP layer (two-layer ELU network per node)
        x = self.mlp1(x)
        # Converts node embeddings to edge embeddings
        x = self.node2edge(x)
        # Passes through the second MLP layer
        x = self.mlp2(x)
        # Converts edge embeddings back to node embeddings
        x = self.edge2node(x)
        x = self.mlp3(x)
        x = self.node2edge(x)
        x = self.mlp4(x)

        # Final fully connected layer to get the logits
        result =  self.fc_out(x)
        result_dict = {
            'logits': result,
            'state': inputs,
            }
        return result_dict

In [9]:
def _to_python_label(label):
    if isinstance(label, torch.Tensor):
        if label.numel() != 1:
            raise ValueError(f"Expected scalar label tensor, got shape {tuple(label.shape)}")
        label = label.item()
    if isinstance(label, np.generic):
        label = label.item()
    return int(label) if isinstance(label, (int, np.integer)) else str(label)


def _extract_label_from_item(item):
    for key in ('label', 'labels', 'target', 'pitch_type', 'pitch_type_idx'):
        if key in item:
            return _to_python_label(item[key])
    raise KeyError(
        "No label key found. Expected one of: "
        "'label', 'labels', 'target', 'pitch_type', 'pitch_type_idx'."
    )


def _pad_or_slice_window(inputs, window_size, start_idx=None):
    if inputs.dim() != 3:
        raise ValueError(
            f"Expected pose tensor shape [time, num_vars, num_feats], got {tuple(inputs.shape)}"
        )
    total_steps = inputs.size(0)

    if total_steps >= window_size:
        if start_idx is None:
            start_idx = total_steps - window_size
        return inputs[start_idx:start_idx + window_size]

    pad_steps = window_size - total_steps
    pad = inputs[0:1].repeat(pad_steps, 1, 1)
    return torch.cat([pad, inputs], dim=0)


def build_label_mapping(dataset):
    labels = []
    for idx in range(len(dataset)):
        labels.append(_extract_label_from_item(dataset[idx]))

    unique_labels = sorted(set(labels))
    label_to_idx = {label: i for i, label in enumerate(unique_labels)}
    idx_to_label = {i: label for label, i in label_to_idx.items()}
    return label_to_idx, idx_to_label


class SlidingWindowPitchDataset(Dataset):
    def __init__(self, base_dataset, window_size=16, label_to_idx=None, training=True):
        self.base_dataset = base_dataset
        self.window_size = window_size
        self.training = training
        if label_to_idx is None:
            raise ValueError('label_to_idx is required for consistent class indexing.')
        self.label_to_idx = label_to_idx

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        item = self.base_dataset[idx]
        inputs = item['inputs'].float()

        total_steps = inputs.size(0)
        if self.training and total_steps > self.window_size:
            start_idx = torch.randint(0, total_steps - self.window_size + 1, (1,)).item()
        else:
            start_idx = max(0, total_steps - self.window_size)

        window = _pad_or_slice_window(inputs, self.window_size, start_idx=start_idx)
        label_raw = _extract_label_from_item(item)
        label_idx = self.label_to_idx[label_raw]

        return {
            'inputs': window,
            'labels': torch.tensor(label_idx, dtype=torch.long),
        }


class NRIPitchTypeClassifier(nn.Module):
    def __init__(self, encoder, num_vars, num_edge_types, num_classes, dropout_prob=0.3):
        super().__init__()
        self.encoder = encoder
        self.num_vars = num_vars
        self.num_edge_types = num_edge_types

        num_edges = num_vars * (num_vars - 1)
        feature_dim = num_edges * num_edge_types
        self.classifier = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, 256),
            nn.ELU(inplace=True),
            nn.Dropout(dropout_prob),
            nn.Linear(256, num_classes),
        )

    def forward(self, inputs):
        logits = self.encoder(inputs)['logits']
        edge_probs = F.softmax(logits, dim=-1)
        flattened_features = edge_probs.reshape(edge_probs.size(0), -1)
        return self.classifier(flattened_features)


@torch.no_grad()
def predict_stream_pitch_types(model, pose_sequence, idx_to_label, window_size=16, stride=1, gpu=True):
    if pose_sequence.dim() != 3:
        raise ValueError(
            f"Expected pose sequence shape [time, num_vars, num_feats], got {tuple(pose_sequence.shape)}"
        )

    model.eval()
    predictions = []
    total_steps = pose_sequence.size(0)

    for end_step in range(1, total_steps + 1, stride):
        start_step = max(0, end_step - window_size)
        window = pose_sequence[start_step:end_step]
        window = _pad_or_slice_window(window, window_size)
        window = window.unsqueeze(0)

        if gpu:
            window = window.cuda(non_blocking=True)

        class_logits = model(window)
        class_probs = torch.softmax(class_logits, dim=1).squeeze(0).cpu()
        pred_idx = int(torch.argmax(class_probs).item())

        predictions.append({
            'frame': end_step - 1,
            'pred_idx': pred_idx,
            'pred_label': idx_to_label[pred_idx],
            'confidence': float(class_probs[pred_idx].item()),
        })

    return predictions

In [10]:
from pathlib import Path

from nri_pose_dataset import load_curated_nri_splits, summarize_dataset_stats

DATASET_ROOT = Path('curated_finetuning_dataset')

train_data, val_data, test_data = load_curated_nri_splits(
    dataset_root=DATASET_ROOT,
    min_frames=8,
    require_image_pair=True,
)

print(summarize_dataset_stats('train', train_data))
print(summarize_dataset_stats('val', val_data))
print(summarize_dataset_stats('test', test_data))

train: sequences=226, parsed_frames=33705, empty_labels=6975, bad_format=0, missing_image_pairs=0, short_sequences=0
val: sequences=30, parsed_frames=4458, empty_labels=942, bad_format=0, missing_image_pairs=0, short_sequences=0
test: sequences=39, parsed_frames=6112, empty_labels=908, bad_format=0, missing_image_pairs=0, short_sequences=0


In [11]:
WINDOW_SIZE = 16
num_vars = 17
num_edge_types = 2

missing_datasets = [name for name in ('train_data', 'val_data', 'test_data') if name not in globals()]
if missing_datasets:
    raise RuntimeError(
        f"Missing dataset variables: {missing_datasets}. Define train_data, val_data, test_data before training."
    )

label_to_idx, idx_to_label = build_label_mapping(train_data)
num_classes = len(label_to_idx)
print('Class mapping:', label_to_idx)

# Build classifier encoder for fixed sliding-window input
encoder = RefMLPEncoder(
    num_vars=num_vars,
    input_size=6,
    input_time_steps=WINDOW_SIZE,
    num_edge_types=num_edge_types,
    encoder_dropout=0.1,
    encoder_hidden=256,
    encoder_mlp_hidden=256,
    )

model = NRIPitchTypeClassifier(
    encoder=encoder,
    num_vars=num_vars,
    num_edge_types=num_edge_types,
    num_classes=num_classes,
    dropout_prob=0.3,
    )
model = model.cuda()

Class mapping: {'CH': 0, 'FF': 1, 'SI': 2, 'SL': 3, 'ST': 4}


In [12]:
gpu = True
batch_size = 8
val_batch_size = batch_size

num_epochs = 60

clip_grad = None
clip_grad_norm = 1.0

train_window_data = SlidingWindowPitchDataset(
    train_data,
    window_size=WINDOW_SIZE,
    label_to_idx=label_to_idx,
    training=True,
    )
val_window_data = SlidingWindowPitchDataset(
    val_data,
    window_size=WINDOW_SIZE,
    label_to_idx=label_to_idx,
    training=False,
    )

train_data_loader = DataLoader(
    train_window_data,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    pin_memory=gpu,
    )
val_data_loader = DataLoader(
    val_window_data,
    batch_size=val_batch_size,
    shuffle=False,
    pin_memory=gpu,
    )

lr = 1e-3
decay_factor = 0.5
decay_steps = 300
wd = 0.0001

In [13]:
def build_scheduler(opt, lr_decay_factor=0.5, lr_decay_steps=300):
    if lr_decay_factor:
        return torch.optim.lr_scheduler.StepLR(opt, lr_decay_steps, lr_decay_factor)
    else:
        return None

In [14]:
model_params = [param for param in model.parameters() if param.requires_grad]
opt = torch.optim.Adam(model_params, lr=lr, weight_decay=wd)
criterion = nn.CrossEntropyLoss()

In [15]:
start_epoch = 1
best_val_epoch = -1
best_val_loss = float('inf')
best_val_acc = 0.0
best_model_state = None

In [16]:
# learning-rate scheduler for classification training
training_scheduler = build_scheduler(opt, lr_decay_factor=decay_factor, lr_decay_steps=decay_steps)

torch.manual_seed(1)
np.random.seed(1)
random.seed(1)

In [17]:
from tqdm import tqdm

pbar = tqdm(range(start_epoch, num_epochs + 1), desc='Epochs')
for epoch in pbar:
    # ---- Training ----
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for batch in train_data_loader:
        inputs = batch['inputs']
        labels = batch['labels']

        if gpu:
            inputs = inputs.cuda(non_blocking=True)
            labels = labels.cuda(non_blocking=True)

        opt.zero_grad()
        class_logits = model(inputs)
        loss = criterion(class_logits, labels)
        loss.backward()

        if clip_grad is not None:
            nn.utils.clip_grad_value_(model.parameters(), clip_grad)
        elif clip_grad_norm is not None:
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad_norm)

        opt.step()

        batch_size_actual = labels.size(0)
        train_loss_sum += loss.item() * batch_size_actual
        preds = class_logits.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += batch_size_actual

    train_loss = train_loss_sum / max(train_total, 1)
    train_acc = train_correct / max(train_total, 1)

    if training_scheduler is not None:
        training_scheduler.step()

    # ---- Validation ----
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for batch in val_data_loader:
            inputs = batch['inputs']
            labels = batch['labels']

            if gpu:
                inputs = inputs.cuda(non_blocking=True)
                labels = labels.cuda(non_blocking=True)

            class_logits = model(inputs)
            loss = criterion(class_logits, labels)

            batch_size_actual = labels.size(0)
            val_loss_sum += loss.item() * batch_size_actual
            preds = class_logits.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += batch_size_actual

    val_loss = val_loss_sum / max(val_total, 1)
    val_acc = val_correct / max(val_total, 1)

    if val_loss < best_val_loss:
        best_val_epoch = epoch
        best_val_loss = val_loss
        best_val_acc = val_acc
        best_model_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }

    pbar.set_postfix({
        'train_loss': f'{train_loss:.4f}',
        'train_acc': f'{train_acc:.4f}',
        'val_loss': f'{val_loss:.4f}',
        'val_acc': f'{val_acc:.4f}',
        'best_val_loss': f'{best_val_loss:.4f}',
    })

if best_model_state is not None:
    model.load_state_dict(best_model_state)
print(f'Best epoch: {best_val_epoch}, val_loss: {best_val_loss:.4f}, val_acc: {best_val_acc:.4f}')

Epochs: 100%|██████████| 60/60 [00:16<00:00,  3.55it/s, train_loss=1.4568, train_acc=0.4018, val_loss=1.2300, val_acc=0.5667, best_val_loss=1.1955]

Best epoch: 58, val_loss: 1.1955, val_acc: 0.5333


In [18]:
def evaluate_pitch_type_classifier(model, dataset, label_to_idx, window_size=16, gpu=True, batch_size=8):
    eval_dataset = SlidingWindowPitchDataset(
        dataset,
        window_size=window_size,
        label_to_idx=label_to_idx,
        training=False,
    )
    data_loader = DataLoader(
        eval_dataset,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=gpu,
    )

    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for batch in data_loader:
            inputs = batch['inputs']
            labels = batch['labels']

            if gpu:
                inputs = inputs.cuda(non_blocking=True)
                labels = labels.cuda(non_blocking=True)

            logits = model(inputs)
            loss = criterion(logits, labels)

            batch_size_actual = labels.size(0)
            total_loss += loss.item() * batch_size_actual
            preds = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_count += batch_size_actual

    return {
        'loss': total_loss / max(total_count, 1),
        'accuracy': total_correct / max(total_count, 1),
        'count': total_count,
    }

In [19]:
mode = 'eval'

if mode == 'eval':
    metrics = evaluate_pitch_type_classifier(
        model,
        test_data,
        label_to_idx=label_to_idx,
        window_size=WINDOW_SIZE,
        gpu=gpu,
        batch_size=batch_size,
    )
    print('PITCH TYPE CLASSIFICATION RESULTS:')
    print(f"\tCross-Entropy Loss: {metrics['loss']:.4f}")
    print(f"\tAccuracy:           {metrics['accuracy']:.4f}")
    print(f"\tSamples:            {metrics['count']}")

    sample = test_data[0]['inputs']
    stream_preds = predict_stream_pitch_types(
        model,
        sample,
        idx_to_label=idx_to_label,
        window_size=WINDOW_SIZE,
        stride=1,
        gpu=gpu,
    )
    print('Last 5 real-time predictions:')
    for pred in stream_preds[-5:]:
        print(
            f"\tframe={pred['frame']}, pred={pred['pred_label']}, conf={pred['confidence']:.3f}"
        )

PITCH TYPE CLASSIFICATION RESULTS:
	Cross-Entropy Loss: 1.5657
	Accuracy:           0.4103
	Samples:            39
Last 5 real-time predictions:
	frame=146, pred=FF, conf=0.625
	frame=147, pred=FF, conf=0.625
	frame=148, pred=FF, conf=0.625
	frame=149, pred=FF, conf=0.648
	frame=150, pred=FF, conf=0.621


In [20]:
from collections import Counter, deque

@torch.no_grad()
def stream_classify_pose_sequence(
    model,
    pose_sequence,
    idx_to_label,
    window_size=16,
    confidence_threshold=0.60,
    smoothing_window=5,
    gpu=True,
    hold_last_on_low_conf=True,
    ):
    """
    Simulate real-time, frame-by-frame classification from a pose sequence tensor.

    Args:
        pose_sequence: torch.Tensor shaped [time, num_vars, num_feats].
    Returns:
        List[dict] with raw prediction, confidence-gated label, and smoothed label per frame.
    """
    if pose_sequence.dim() != 3:
        raise ValueError(
            f"Expected pose sequence shape [time, num_vars, num_feats], got {tuple(pose_sequence.shape)}"
        )

    model.eval()
    frame_buffer = deque(maxlen=window_size)
    smoothing_buffer = deque(maxlen=smoothing_window)
    results = []

    stable_label = None

    for frame_idx in range(pose_sequence.size(0)):
        frame = pose_sequence[frame_idx]
        frame_buffer.append(frame)

        # Build the current inference window from the rolling buffer.
        current_window = torch.stack(list(frame_buffer), dim=0)
        current_window = _pad_or_slice_window(current_window, window_size)
        model_input = current_window.unsqueeze(0)

        if gpu:
            model_input = model_input.cuda(non_blocking=True)

        logits = model(model_input)
        probs = torch.softmax(logits, dim=1).squeeze(0).cpu()
        pred_idx = int(torch.argmax(probs).item())
        pred_label = idx_to_label[pred_idx]
        confidence = float(probs[pred_idx].item())

        if confidence >= confidence_threshold:
            accepted_label = pred_label
            stable_label = pred_label
            smoothing_buffer.append(accepted_label)
            is_low_conf = False
        else:
            is_low_conf = True
            if hold_last_on_low_conf and stable_label is not None:
                accepted_label = stable_label
                smoothing_buffer.append(accepted_label)
            else:
                accepted_label = 'LOW_CONF'

        if len(smoothing_buffer) > 0:
            smoothed_label = Counter(smoothing_buffer).most_common(1)[0][0]
        else:
            smoothed_label = accepted_label

        results.append({
            'frame': frame_idx,
            'raw_label': pred_label,
            'accepted_label': accepted_label,
            'smoothed_label': smoothed_label,
            'confidence': confidence,
            'low_confidence': is_low_conf,
        })

    return results


# Example usage on one sample sequence from test split
stream_example = test_data[0]['inputs']
stream_results = stream_classify_pose_sequence(
    model=model,
    pose_sequence=stream_example,
    idx_to_label=idx_to_label,
    window_size=WINDOW_SIZE,
    confidence_threshold=0.60,
    smoothing_window=5,
    gpu=gpu,
    hold_last_on_low_conf=True,
    )

print('Last 10 streaming outputs:')
for row in stream_results[-10:]:
    print(
        f"frame={row['frame']:>3} | raw={row['raw_label']:<14} | "
        f"accepted={row['accepted_label']:<14} | "
        f"smoothed={row['smoothed_label']:<14} | conf={row['confidence']:.3f}"
    )

Last 10 streaming outputs:
frame=141 | raw=FF             | accepted=FF             | smoothed=FF             | conf=0.625
frame=142 | raw=FF             | accepted=FF             | smoothed=FF             | conf=0.625
frame=143 | raw=FF             | accepted=FF             | smoothed=FF             | conf=0.625
frame=144 | raw=FF             | accepted=FF             | smoothed=FF             | conf=0.625
frame=145 | raw=FF             | accepted=FF             | smoothed=FF             | conf=0.625
frame=146 | raw=FF             | accepted=FF             | smoothed=FF             | conf=0.625
frame=147 | raw=FF             | accepted=FF             | smoothed=FF             | conf=0.625
frame=148 | raw=FF             | accepted=FF             | smoothed=FF             | conf=0.625
frame=149 | raw=FF             | accepted=FF             | smoothed=FF             | conf=0.648
frame=150 | raw=FF             | accepted=FF             | smoothed=FF             | conf=0.621
